# Figure-5-style Pareto trade-off across alpha

Reproduces the experiment design of Figure 5 from Holliday, El-Geneidy &
Dudek (2025) "Learning heuristics for transit network design ...": for each
value of `alpha in [0, 1]` we set the cost weights to
`(demand_time_weight=alpha, route_time_weight=1-alpha,
median_connectivity_weight=0)` and plot the resulting `(ATT, RTT)` = `(C_p, C_o)`
point per method. Connecting the points in `alpha` order yields the
trade-off curve -- down-and-leftward means a strictly better network.

The headline ablation is the **Figure-5-style isolation of RL-trained
construction**: compare `neural_extend_trim_split_5_5` (paper's NEA analog --
5 trained-construction bees + 5 trained-edit/trim bees) against
`path_combiner_extend_trim_split_5_5` (paper's RC-EA analog -- 5
RandomPathCombiningRouteGenerator bees + 5 trained-edit/trim bees). Both
share the same trained edit head, so the gap between their curves measures
the contribution of *trained* construction on top of random path
composition.

This notebook is fully self-contained: configuration -> sweep -> CSV save ->
three styles of figure. Re-running drops earlier results unless
`REUSE_EXISTING_PARETO_SWEEP = True` is set and `pareto_alpha_sweep` is
already in scope. Persistent artifacts:

* `artifacts/results/pareto_alpha_sweep_rows.csv` -- per-run rows
  (city x method x accept_mode x alpha x seed) with a flat metric set.
* `artifacts/results/pareto_alpha_sweep_summary.csv` -- seed-aggregated
  mean/std grouped by (city, method, accept_mode, alpha).
* Figures show in the notebook output; the underlying data is regenerable
  from the CSVs after a kernel restart.

## 1. Imports

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# eval_lib must be importable -- the notebook's working directory holds it.
import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *  # noqa: F401,F403

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("eval_lib OK; RESULTS_DIR =", RESULTS_DIR.relative_to(ROOT_DIR))
print(f"Available BCO variants ({len(BCO_VARIANTS)}):")
for v in BCO_VARIANTS:
    print(f"  {v['key']}")

## 2. Configuration

Adjust the constants below and re-run. Defaults: 5-point `alpha` grid, Mandl
+ Mumford0 (the two cheapest benchmark cities), 1 seed, `without_worse`
accept mode only. Set `PARETO_SEEDS = list(range(3))` for narrow std bars on
the Pareto figure; broaden to more cities or 11-point `alpha` for a full
paper-style reproduce.

In [ ]:
# What to vary -- single source of truth.
PARETO_CITIES = ["Mandl", "Mumford0"]
PARETO_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
PARETO_SEEDS = [int(WORSE_ACCEPT_SEEDS[0]) if WORSE_ACCEPT_SEEDS else 0]
PARETO_ACCEPT_MODES = ("without_worse",)
REUSE_EXISTING_PARETO_SWEEP = True

# Figure-5 ablation variants live ONLY here, not in the default BCO_VARIANTS
# (which would pollute the §12 benchmark sweep / MACSA / worse-accept tables
# with rows that only make sense in the Pareto-alpha sweep). Both pair the
# same trained edit/trim head (5 type-5 bees) with two different construction
# halves -- trained vs random path combiner -- so the gap between their
# curves directly isolates the contribution of RL-trained construction over
# untrained shortest-path composition.
PARETO_ABLATION_VARIANTS = [
    {
        # NEA-equivalent: 5 type-1 trained-construction bees (bestsofar_feb2023)
        # + 5 type-5 trained-edit/trim bees (bestsofar_feb2023_trim).
        "key": "neural_extend_trim_split_5_5",
        "summary_label": "Neural rebuild + extend/trim edit BCO (5+5)",
        "run_name": "seeded_bco_neural_extend_trim_split_5_5_from_lc_mumford0",
        "use_neural_bees": True,
        "n_type1_bees": BCO_N_TYPE1_BEES,
        "n_type2_bees": 0,
        "n_type4_bees": 0,
        "n_type5_bees": BCO_N_BEES - BCO_N_TYPE1_BEES,
        "n_type6_bees": 0,
        "n_type7_bees": 0,
    },
    {
        # RC-EA-equivalent: 5 type-3 bees auto-instantiate a
        # RandomPathCombiningRouteGenerator (paper's pi_random) -- n_type3 is
        # derived as n_bees - sum(other types) = 10 - 0 - 0 - 0 - 5 - 0 - 0 = 5
        # inside bee_colony.get_neural_variants. use_neural_bees=False because
        # bee_colony spins its own RPC model and does not need
        # MODEL_WEIGHTS_PATH to be loaded.
        "key": "path_combiner_extend_trim_split_5_5",
        "summary_label": "Path-combiner rebuild + extend/trim edit BCO (5+5)",
        "run_name": "seeded_bco_path_combiner_extend_trim_split_5_5_from_lc_mumford0",
        "use_neural_bees": False,
        "n_type1_bees": 0,
        "n_type2_bees": 0,
        "n_type4_bees": 0,
        "n_type5_bees": BCO_N_BEES - BCO_N_TYPE1_BEES,
        "n_type6_bees": 0,
        "n_type7_bees": 0,
    },
]

# Combined variant list for this sweep only: library defaults + the two
# Figure-5 ablation variants. Used everywhere below; library BCO_VARIANTS
# stays untouched so other notebooks see only the standard 6 variants.
PARETO_BCO_VARIANTS = list(BCO_VARIANTS) + PARETO_ABLATION_VARIANTS

# Method axis = Initial + RL-only + every variant from the combined list.
PARETO_METHOD_SPECS = (
    [INITIAL_METHOD]
    + ([RL_ONLY_METHOD] if RUN_RL_ONLY_BASELINE else [])
    + [bco_method(v) for v in PARETO_BCO_VARIANTS]
)

# Figure-5 ablation labels -- pulled from the ablation list directly so we
# never depend on these keys being in the default BCO_VARIANTS.
NEA_VARIANT_LABEL = next(
    (v["summary_label"] for v in PARETO_ABLATION_VARIANTS
     if v["key"] == "neural_extend_trim_split_5_5"), None)
RCEA_VARIANT_LABEL = next(
    (v["summary_label"] for v in PARETO_ABLATION_VARIANTS
     if v["key"] == "path_combiner_extend_trim_split_5_5"), None)
print(f"NEA-style label (paper Figure 5):  {NEA_VARIANT_LABEL}")
print(f"RC-EA-style label (paper Figure 5): {RCEA_VARIANT_LABEL}")

print(f"\nDefault BCO_VARIANTS ({len(BCO_VARIANTS)}):")
for v in BCO_VARIANTS:
    print(f"  {v['key']}")
print(f"\nPareto-only ablation variants ({len(PARETO_ABLATION_VARIANTS)}):")
for v in PARETO_ABLATION_VARIANTS:
    print(f"  {v['key']}")

print(f"\nSweep matrix: "
      f"{len(PARETO_CITIES)} cities x {len(PARETO_ALPHAS)} alphas x "
      f"{len(PARETO_METHOD_SPECS)} methods x {len(PARETO_SEEDS)} seeds = "
      f"{len(PARETO_CITIES) * len(PARETO_ALPHAS) * len(PARETO_METHOD_SPECS) * len(PARETO_SEEDS)} runs "
      f"(BCO methods only fan over seeds; initial / rl_only run once per (city, alpha))")

## 3. Build per-city specs

For each city in `PARETO_CITIES` we build the **same NX-heuristic initial
network** the section-12 benchmark sweep uses, so points at `alpha ~ 0.33`
are directly comparable with the corresponding row in
`benchmark_sweep.csv`.

In [ ]:
PARETO_CITY_SPECS = []
for spec in BENCHMARK_SPECS:
    if spec["city"] not in PARETO_CITIES:
        continue
    tensors, init_routes = load_benchmark_graph(spec)
    PARETO_CITY_SPECS.append({
        "dataset": spec["city"],
        "init_routes": init_routes,
        "tensors": tensors,
        "n_routes": spec["n_routes"],
        "min_route_len": spec["min_route_len"],
        "max_route_len": spec["max_route_len"],
    })

for spec in PARETO_CITY_SPECS:
    n_nodes = int(spec["tensors"]["node_locs"].shape[0])
    print(f"  {spec['dataset']:<10} n_nodes={n_nodes} "
          f"n_routes={spec['n_routes']} "
          f"min={spec['min_route_len']} max={spec['max_route_len']}")

## 4. Run the alpha-Pareto sweep

`run_alpha_pareto_sweep` (in `eval_lib/sweep.py`) wraps `run_seed_sweep` and
re-runs the city x method x seed grid once per `alpha`, tagging each row
with its `alpha`. Both raw `rows_df` and seed-aggregated `summary_df` are
persisted as CSV so figures stay reconstructible across kernel restarts.

In [ ]:
if REUSE_EXISTING_PARETO_SWEEP and "pareto_alpha_sweep" in globals():
    print("Reusing existing pareto_alpha_sweep; "
          "set REUSE_EXISTING_PARETO_SWEEP=False to rerun.")
else:
    pareto_alpha_sweep = run_alpha_pareto_sweep(
        PARETO_CITY_SPECS, PARETO_METHOD_SPECS, PARETO_ALPHAS,
        seeds=PARETO_SEEDS, accept_modes=PARETO_ACCEPT_MODES,
        run_name_scope="pareto_")

pareto_rows_df = pareto_alpha_sweep["rows_df"]
pareto_summary_df = pareto_alpha_sweep["summary_df"]
save_table(pareto_rows_df, "pareto_alpha_sweep_rows")
save_table(pareto_summary_df, "pareto_alpha_sweep_summary")
print(f"rows_df: {len(pareto_rows_df)} rows, "
      f"{len(pareto_rows_df['method'].unique())} methods, "
      f"alphas={sorted(pareto_rows_df['alpha'].unique())}")
display(pareto_rows_df.head(10))

## 5. Pareto trade-off figure -- C_p vs C_o per city

The Figure-5 / Figure-3 paper shape: one subplot per city, one curve per
method, points connected in `alpha` order. Down-and-left means strictly
better. Single-seed runs give zero-width error bars; with >1 seed each
point shows mean +/- std.

In [ ]:
method_labels = [m.get("label", m["kind"]) for m in PARETO_METHOD_SPECS]
fig = plot_alpha_pareto_grid(
    pareto_rows_df,
    cities=[s["dataset"] for s in PARETO_CITY_SPECS],
    methods=method_labels,
    title_prefix="Pareto trade-off across alpha (ATT vs RTT)")
plt.show()

## 6. Cost vs alpha per city

A complementary view to the Pareto: one curve per method, `x = alpha`,
`y = weighted cost`. Makes it explicit which method is best at which alpha
value -- the Pareto plot collapses this information into one curve via the
`(C_p, C_o)` projection.

In [ ]:
cities_list = [s["dataset"] for s in PARETO_CITY_SPECS]
n_cities = len(cities_list)
n_cols = min(n_cities, 2)
n_rows = math.ceil(n_cities / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, squeeze=False,
                          figsize=(7.0 * n_cols, 4.8 * n_rows))
cmap = plt.get_cmap("tab10")
method_colors = {m: cmap(i % 10) for i, m in enumerate(method_labels)}

for idx, city in enumerate(cities_list):
    ax = axes[idx // n_cols, idx % n_cols]
    city_df = pareto_rows_df[pareto_rows_df["dataset"] == city]
    for method in method_labels:
        mdf = city_df[city_df["method"] == method]
        if mdf.empty:
            continue
        agg = (mdf.groupby("alpha")["cost"]
                  .agg(["mean", "std"])
                  .reset_index()
                  .sort_values("alpha"))
        ax.errorbar(agg["alpha"], agg["mean"], yerr=agg["std"].fillna(0),
                    marker="o", linestyle="-", label=method,
                    color=method_colors[method], capsize=2, alpha=0.85)
    ax.set_xlabel("alpha (0 = operator, 1 = passenger)")
    ax.set_ylabel("weighted cost (lower = better)")
    ax.set_title(city, fontweight="bold")
    ax.grid(alpha=0.25)
for j in range(n_cities, n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis("off")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center",
            bbox_to_anchor=(0.5, -0.05), ncol=min(4, len(labels)),
            frameon=True)
fig.suptitle("Cost vs alpha by method", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
plt.show()

## 7. Figure-5 ablation -- trained construction vs random path combiner

Isolated comparison of the two ablation variants:

* **`neural_extend_trim_split_5_5`** (NEA-equivalent): 5 type-1 trained-
  construction bees + 5 type-5 trained-edit/trim bees -- both halves use
  RL-trained models.
* **`path_combiner_extend_trim_split_5_5`** (RC-EA-equivalent): 5 type-3
  random-path-combiner bees (paper's pi_random) + 5 type-5 trained-edit/
  trim bees -- only the edit head is trained.

`extend_trim_edit_only` is shown as a third reference (all 10 bees on the
trained edit head, no construction half), so the gap between the three
isolates the contribution of the construction half independently of the
edit head.

The figure is the same Pareto shape as section 5 but filtered to these
three curves.

In [ ]:
# extend_trim_edit_only is the "no construction half" reference, still in
# the default BCO_VARIANTS. NEA/RC-EA labels come from the inline
# PARETO_ABLATION_VARIANTS defined in the config cell.
ABLATION_METHODS = [m for m in [
    NEA_VARIANT_LABEL, RCEA_VARIANT_LABEL,
    next((v["summary_label"] for v in BCO_VARIANTS
          if v["key"] == "extend_trim_edit_only"), None),
] if m is not None]
print(f"Ablation curves: {ABLATION_METHODS}")

fig = plot_alpha_pareto_grid(
    pareto_rows_df[pareto_rows_df["method"].isin(ABLATION_METHODS)].copy(),
    cities=cities_list,
    methods=ABLATION_METHODS,
    title_prefix="Figure-5 ablation: trained construction vs random path combiner")
plt.show()

# Save the filtered subset as its own CSV so the ablation curves can be
# rebuilt without re-deriving the filter logic.
ablation_df = pareto_rows_df[
    pareto_rows_df["method"].isin(ABLATION_METHODS)].copy()
save_table(ablation_df, "pareto_alpha_sweep_ablation_rows")

## 8. Per-alpha summary tables

Seed-aggregated mean cost / ATT / RTT / d_un per `(city, alpha, method)`,
shown as one table per `alpha`. Useful for citing specific numbers in
write-ups.

In [ ]:
for alpha in PARETO_ALPHAS:
    alpha_df = pareto_summary_df[pareto_summary_df["alpha"] == alpha]
    if alpha_df.empty:
        continue
    print(f"\n=== alpha = {alpha} ===")
    show_cols = [c for c in (
        "dataset", "method", "kind", "n_seeds",
        "mean_cost", "std_cost", "mean_ATT", "mean_RTT",
        "mean_$d_{un}$")
        if c in alpha_df.columns]
    display(alpha_df[show_cols].sort_values(["dataset", "mean_cost"]).reset_index(drop=True))